# Loopback latency: the same measurement on the board and on the emulator

A const pulse goes out of generator `GEN_CH`, through a loopback cable (DAC -> ADC), and is read by readout `RO_CH` twice: with the decimated
readout (window opened at `t = 0`) and with the multi-rate (full-speed) buffer. The latency is the time at which the signal reaches half of its
maximum, on the time axis the QICK software gives to the data. The definition is the one of steps 2 and 7 of
the tProc v2 validation notebook of the RFSoC4x2 debug bench, so the numbers can be put next to the RFSoC4x2 ones
(decimated 0.293 to 0.295 us, multi-rate buffer 0.160 us).

* `ON_BOARD = False` (default): runs on the emulator. It does not need a board.
* `ON_BOARD = True`: run it on the board (PYNQ), with the bitstream of `BITFILE`, and copy the printed `RESULT` line. The cable must join the DAC of
  generator `GEN_CH` to the ADC of readout `RO_CH` (the printed `soccfg` says which tiles those are). The pulse is a 100 MHz tone of gain 0.5, so
  nothing is saturated.

ZCU216 with `tpv2_std-v2.4.377-C817EB7` (`Top/216/tpv2_std` at the commit of #123, tProc `c_clk` 230.4 MHz): generator 0 is `axis_signal_gen_v6_0` on DAC tile 0 block 0
(`0_228`) and readout 0 is `axis_dyn_readout_v1_0` on ADC tile 2 block 0 (`0_226`), so the cable goes from DAC `0_228` to ADC `0_226`, through
baluns that pass 100 MHz. Check it against the tiles in the printed `soccfg` before measuring.


In [ ]:
ON_BOARD = False
BITFILE  = "/home/xilinx/jupyter_notebooks/fw/tpv2_std-v2.4.377-C817EB7.bit"   # only used with ON_BOARD = True (the .hwh next to it)
QICK_LIB = "/home/xilinx/jupyter_notebooks/qick/qick_lib"   # ON_BOARD: the clone of this repository on the board
GEN_CH, RO_CH = 0, 0
FREQ, PULSE_LEN, GAIN = 100.0, 0.3, 0.5     # MHz, us

import sys, pathlib
if not ON_BOARD:
    QICK_LIB = str(pathlib.Path.cwd().parent.parent / 'qick_lib')      # this repository
# Always at the front: the pynq-venv can have its own qick (an egg) ahead of it, without the qick:ip drivers.
sys.path[:] = [QICK_LIB] + [p for p in sys.path if p != QICK_LIB]
import qick
print("qick", qick.__version__, "from", qick.__file__)
assert qick.__file__.startswith(QICK_LIB), "qick was imported from %s: restart the kernel and run this cell first" % qick.__file__

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from qick.asm_v2 import AveragerProgramV2


In [ ]:
if ON_BOARD:
    from qick import QickSoc
    soc = QickSoc(BITFILE)
    soccfg = soc
else:
    REPO_ROOT = pathlib.Path.cwd().parent          # emulator/
    sys.path.insert(0, str(REPO_ROOT / 'software'))
    from qick_emu import QickEmu
    soc = QickEmu(str(REPO_ROOT / 'config' / 'qick_emu_config.json'))
    soccfg = soc.soccfg
    OUT = REPO_ROOT / 'artifacts' / 'loopback_latency'
    OUT.mkdir(parents=True, exist_ok=True)
print(soccfg)

In [ ]:
class PulseProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])
        self.add_pulse(ch=cfg['gen_ch'], name="p", style="const", freq=cfg['freq'], ro_ch=cfg['ro_ch'],
                       length=cfg['pulse_len'], phase=0, gain=cfg['gain'])
        self.add_readoutconfig(ch=cfg['ro_ch'], name="ro", freq=cfg['freq'], gen_ch=cfg['gen_ch'])
        self.send_readoutconfig(ch=cfg['ro_ch'], name="ro", t=0)

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="p", t=cfg['pulse_t'])
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=0, mr=cfg['mr'])

base = dict(gen_ch=GEN_CH, ro_ch=RO_CH, freq=FREQ, ro_len=1.5, pulse_len=PULSE_LEN, gain=GAIN, pulse_t=0.0, mr=False)

def prepare(name, **kw):
    if not ON_BOARD:
        d = OUT / name; d.mkdir(exist_ok=True)
        global soc
        soc = soc.prepare_emu(memdir=d, build=False, trace=False, verbose=False, **kw)
        return d

def half_max_start(t, mag):
    above = t[mag >= 0.5 * mag.max()]
    return above[0], above[-1] - above[0]

## Decimated readout (window opened at t = 0, pulse at t = 0)

In [ ]:
prepare('dec')
prog = PulseProgram(soccfg, reps=1, final_delay=1.0, cfg=base)
iq = prog.acquire_decimated(soc, rounds=10 if ON_BOARD else 1)[0]
t = prog.get_time_axis(ro_index=0)
mag = np.abs(iq[:, 0] + 1j * iq[:, 1])
t_dec, w_dec = half_max_start(t, mag)
plt.plot(t, mag); plt.xlabel("us"); plt.ylabel("|IQ|"); plt.show()
print(f"decimated: peak {mag.max():.1f}, starts at {t_dec:.4f} us, half-max width {w_dec:.4f} us")

## Multi-rate (full-speed) buffer (pulse at t = 0.1 us, window opened at t = 0)

In [ ]:
cfg = {**base, 'pulse_t': 0.1, 'mr': True}
prog = PulseProgram(soccfg, reps=1, final_delay=1.0, cfg=cfg)
if ON_BOARD:
    soc.arm_mr(ch=RO_CH)
    prog.acquire(soc, rounds=1)
    iq_mr = soc.get_mr()
else:
    d = prepare('mr', mr_len=8192)
    prog.acquire(soc, rounds=1)
    iq_mr = soc.load_mr(d)[1]
t_mr = prog.get_time_axis_mr(RO_CH, iq_mr)
mag = np.abs(iq_mr[:, 0] + 1j * iq_mr[:, 1].astype(float))
k = max(1, int(round(0.01 * len(t_mr) / (t_mr[-1] - t_mr[0]))))      # smooth: the tone is at the mixed-down rate
env = np.convolve(mag, np.ones(k) / k, mode='same')
t0, w_mr = half_max_start(t_mr, env)
plt.plot(t_mr, env); plt.xlim(0, 1.0); plt.xlabel("us"); plt.ylabel("|IQ| smoothed"); plt.show()
lat_mr = t0 - cfg['pulse_t']
print(f"multi-rate buffer: pulse (sent at {cfg['pulse_t']} us) starts at {t0:.4f} us -> latency {lat_mr:.4f} us, half-max width {w_mr:.4f} us")

## Result

In [ ]:
print("RESULT", dict(where='board' if ON_BOARD else 'emulator', decimated_us=round(float(t_dec), 4), decimated_width_us=round(float(w_dec), 4),
                     mr_latency_us=round(float(lat_mr), 4), mr_width_us=round(float(w_mr), 4)))